<a href="https://colab.research.google.com/github/souravk03/Machine-Learning-404/blob/main/Column_Transformer_Day_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets/covid_toy.csv')
df.sample(5)

,age,gender,fever,cough,city,has_covid
98,5,Female,98.0,Strong,Mumbai,No
13,64,Male,102.0,Mild,Bangalore,Yes
69,73,Female,103.0,Mild,Delhi,No
60,24,Female,102.0,Strong,Bangalore,Yes
79,48,Female,103.0,Mild,Kolkata,Yes


In [4]:
df.isnull().sum()


,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

In [9]:
xtrain,xtest,ytrain,ytest=train_test_split(df.iloc[:,:5],df.iloc[:,-1],test_size=0.2)
xtrain

,age,gender,fever,cough,city
69,73,Female,103.0,Mild,Delhi
10,75,Female,NaN,Mild,Delhi
31,83,Male,103.0,Mild,Kolkata
4,65,Female,101.0,Mild,Mumbai
78,11,Male,100.0,Mild,Bangalore
...,...,...,...,...,...
99,10,Female,98.0,Strong,Kolkata
11,65,Female,98.0,Mild,Mumbai
19,42,Female,NaN,Strong,Bangalore
8,19,Female,100.0,Strong,Bangalore


#Without Column Transformer

In [13]:
'''Handling missing values first using imouter'''
si=SimpleImputer()
xtrain_fever=si.fit_transform(xtrain[['fever']])
# for testing data too

xtest_fever=si.fit_transform(xtest[['fever']])
xtrain_fever.shape

(80, 1)

## Nomical encoding for Gender and city and Ordinal encoding for Cough

In [16]:
#ordinal encoder
Oe=OrdinalEncoder(categories=[['Mild','Strong']])
xtrain_cough=Oe.fit_transform(xtrain[['cough']])
xtest_cough=Oe.fit_transform(xtest[['cough']])
xtrain_cough.shape

(80, 1)

In [25]:
# OneHotEncoder
Ohe=OneHotEncoder(drop='first',sparse_output=True)

xtrain_gender_city=Ohe.fit_transform(xtrain[['gender','city']])
xtest_gender_city=Ohe.fit_transform(xtest[['gender','city']])


In [28]:
# Extracting Age
xtrain_age = xtrain.drop(columns=['gender','fever','cough','city']).values

# also the test data
xtest_age = xtest.drop(columns=['gender','fever','cough','city']).values

xtrain_age.shape

(80, 1)

In [30]:
X_train_transformed = np.concatenate((xtrain_age,xtrain_fever,xtrain_gender_city.toarray(),xtrain_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((xtest_age,xtest_fever,xtest_gender_city.toarray(),xtest_cough),axis=1)

X_train_transformed.shape

(80, 7)

#With Using ***Column Transformer***

In [32]:
from sklearn.compose import ColumnTransformer

In [33]:
transformer=ColumnTransformer(transformers=[
    ("tnf1",SimpleImputer(),['fever']),
     ("tnf2",OneHotEncoder(drop="first",sparse_output=True),['gender','city']),
      ("tnf3",OrdinalEncoder(categories=[['Mild','Strong']]),['cough'])],remainder="passthrough")

In [35]:
transformer.fit_transform(xtrain).shape

(80, 7)

In [40]:
transformer.fit_transform(xtest)


array([[103.   ,   0.   ,   0.   ,   1.   ,   0.   ,   0.   ,  48.   ],
       [ 98.   ,   1.   ,   0.   ,   0.   ,   0.   ,   1.   ,  12.   ],
       [104.   ,   0.   ,   1.   ,   0.   ,   0.   ,   1.   ,  34.   ],
       [101.   ,   0.   ,   0.   ,   0.   ,   0.   ,   0.   ,  38.   ],
       [100.875,   0.   ,   0.   ,   0.   ,   1.   ,   1.   ,  34.   ],
       [101.   ,   1.   ,   0.   ,   0.   ,   0.   ,   1.   ,  47.   ],
       [104.   ,   1.   ,   0.   ,   0.   ,   1.   ,   0.   ,  42.   ],
       [ 99.   ,   0.   ,   1.   ,   0.   ,   0.   ,   1.   ,  59.   ],
       [ 98.   ,   1.   ,   1.   ,   0.   ,   0.   ,   0.   ,  83.   ],
       [ 99.   ,   0.   ,   0.   ,   0.   ,   0.   ,   1.   ,  49.   ],
       [100.875,   1.   ,   0.   ,   1.   ,   0.   ,   1.   ,  71.   ],
       [100.   ,   1.   ,   0.   ,   0.   ,   0.   ,   0.   ,  80.   ],
       [100.   ,   0.   ,   0.   ,   0.   ,   0.   ,   1.   ,  47.   ],
       [101.   ,   0.   ,   1.   ,   0.   ,   0.   ,   1.   ,  6